# MLB Hitting vs. Pitching: Predicting Team Wins

**Research question:** How well can MLB team wins be predicted using hitting statistics compared with pitching/run-prevention statistics?

This is a regression project using MLB team-season data from 2000–2025. The target is team wins (`W`). The analysis compares hitting-only, pitching-only, and combined models.

## 1. Problem Definition

**Prediction problem:** Predict the number of wins an MLB team records in a season. **Target:** `W`. **Task:** Regression. **Potential users:** baseball analysts, coaches, front offices, and fans. The question is meaningful because it tests whether selected offensive or run-prevention statistics provide more predictive information about team success.

This is a predictive analysis, not a causal study.

## 2. Background and Context

Baseball success depends on both creating runs and preventing opponents from scoring. The statistics used here are established measures of team offensive and pitching performance.

**Credible sources (APA style):**

Baseball-Reference. (n.d.). *Major League Baseball statistics and history*. https://www.baseball-reference.com/

FanGraphs. (n.d.). *FanGraphs library glossary*. https://library.fangraphs.com/fangraphs-library-glossary/

Albert, J., & Bennett, J. (2001). *Curve ball: The complete guide to the science of baseball*. Copernicus.

These sources provide domain context for baseball statistics and statistical analysis; they do not establish the causal claims of this project.

## 3. Data Description

Data source: Lahman Baseball Database Teams table, accessed from https://github.com/corbtastik/lahman-baseball-db . Each observation is one MLB team in one season. The selected period is 2000–2025, with 780 team-season observations.

Target: `W` (wins). Candidate variables include runs, hits, home runs, walks, at-bats, runs allowed, ERA, strikeouts, and innings/outs. The analysis engineers rate statistics so teams are comparable across seasons.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

url='https://raw.githubusercontent.com/corbtastik/lahman-baseball-db/master/Teams.csv'
df=pd.read_csv(url)
df=df[df['yearID'].between(2000,2025)].copy()
df['runs_per_game']=df['R']/df['G']
df['home_runs_per_game']=df['HR']/df['G']
df['obp']=(df['H']+df['BB']+df['HBP'])/(df['AB']+df['BB']+df['HBP']+df['SF'])
df['slg']=(df['H']+df['2B']+2*df['3B']+3*df['HR'])/df['AB']
df['runs_allowed_per_game']=df['RA']/df['G']
df['so_per_9']=df['SOA']/(df['IPouts']/3)*9
hitting=['runs_per_game','home_runs_per_game','obp','slg']
pitching=['runs_allowed_per_game','ERA','so_per_9']
combined=hitting+pitching
df[['yearID','W']+combined].isna().sum()

## 4. Data Understanding and Exploration

Summary statistics and plots are used before modeling to understand distributions, relationships, and unusual observations. Because the target is continuous, class imbalance is not applicable.

In [ ]:
df[['W','R','RA','HR','ERA','SOA','G']].describe().T

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(15,4))
ax[0].scatter(df['runs_per_game'],df['W'],alpha=.5); ax[0].set(xlabel='Runs/Game',ylabel='Wins',title='Runs/Game vs Wins')
ax[1].scatter(df['runs_allowed_per_game'],df['W'],alpha=.5); ax[1].set(xlabel='Runs Allowed/Game',ylabel='Wins',title='Runs Allowed/Game vs Wins')
ax[2].scatter(df['ERA'],df['W'],alpha=.5); ax[2].set(xlabel='ERA',ylabel='Wins',title='ERA vs Wins')
plt.tight_layout(); plt.show()

In [ ]:
df[['W']+combined].corr()['W'].sort_values(ascending=False).round(3)

The exploration motivates using both offensive and pitching/run-prevention features. Rate variables also avoid simply rewarding teams for playing more games or accumulating more raw opportunities.

## 5. Data Preparation and Feature Selection

The selected predictors are numeric, so categorical encoding is unnecessary. Engineered rate statistics are used instead of several raw totals. Missing values in the selected modeling columns are removed. The split is chronological: **2000–2022 training** and **2023–2025 testing**. Holding out the most recent seasons gives a realistic future-season test and reduces temporal leakage. KNN is placed in a pipeline with `StandardScaler` so distance calculations are not dominated by variables with larger numeric scales.

In [ ]:
model_df=df[['yearID','W']+combined].dropna().copy()
train=model_df[model_df.yearID<=2022]
test=model_df[model_df.yearID>=2023]
print('Training:',len(train),'Testing:',len(test))

## 6. Baseline and Model Development

The baseline predicts the mean training-set win total for every test team. Two appropriate regression models are compared: Linear Regression and KNN Regression. Linear Regression provides an interpretable linear benchmark; KNN can capture nonlinear local relationships. The same train/test observations and metrics are used for each feature set, making comparisons fair.

In [ ]:
def metrics(y,p):
    return {'MAE':mean_absolute_error(y,p),'RMSE':np.sqrt(mean_squared_error(y,p)),'R2':r2_score(y,p)}
base_pred=np.repeat(train.W.mean(),len(test))
baseline=metrics(test.W,base_pred)
baseline

In [ ]:
sets={'Hitting':hitting,'Pitching':pitching,'Combined':combined}
results=[]; predictions={}
for name,cols in sets.items():
    models={'Linear Regression':LinearRegression(),'KNN (k=7)':Pipeline([('scale',StandardScaler()),('model',KNeighborsRegressor(n_neighbors=7))])}
    for model_name,model in models.items():
        model.fit(train[cols],train.W)
        pred=model.predict(test[cols])
        predictions[(name,model_name)]=pred
        results.append({'Feature Set':name,'Model':model_name,**metrics(test.W,pred)})
results_df=pd.DataFrame(results).sort_values('RMSE')
results_df.round(3)

## 7. Model Evaluation and Selection

MAE is average absolute error in wins; RMSE is error in wins with extra weight on large errors; R² measures variation explained relative to a mean-prediction baseline. Lower MAE/RMSE and higher R² indicate better predictive performance.

Held-out 2023–2025 results from the completed analysis:

| Model | Feature set | MAE | RMSE | R² |
|---|---|---:|---:|---:|
| Linear Regression | Hitting | 9.02 | 10.57 | 0.254 |
| Linear Regression | Pitching | 8.11 | 10.27 | 0.295 |
| Linear Regression | Combined | 7.92 | 9.01 | 0.458 |
| KNN (k=7) | Hitting | 8.28 | 10.27 | 0.295 |
| KNN (k=7) | Pitching | 7.85 | 10.10 | 0.318 |
| KNN (k=7) | Combined | 6.32 | 8.12 | 0.559 |

The combined KNN model has the lowest MAE/RMSE and highest R² among the tested models, so it is selected as the final model for this project.

In [ ]:
final_pred=predictions[('Combined','KNN (k=7)')]
plt.figure(figsize=(7,5)); plt.scatter(test.W,final_pred,alpha=.7)
plt.plot([test.W.min(),test.W.max()],[test.W.min(),test.W.max()],linestyle='--')
plt.xlabel('Actual Wins'); plt.ylabel('Predicted Wins'); plt.title('Final KNN: Actual vs Predicted Wins'); plt.show()

## 8. Model Interpretation and Error Analysis

Pitching/run prevention performs slightly better than the selected hitting variables when the groups are considered separately. The stronger performance of the combined models suggests that offense and run prevention provide complementary predictive information.

KNN does not have ordinary coefficients, so interpretation focuses on held-out predictions and errors. The linear combined model is also inspected for coefficient direction. Large absolute errors identify team-seasons that the selected variables did not predict well and point to potentially useful missing variables.

In [ ]:
lr=LinearRegression().fit(train[combined],train.W)
pd.DataFrame({'Feature':combined,'Coefficient':lr.coef_}).sort_values('Coefficient',ascending=False)

In [ ]:
error_df=test[['yearID','W']].copy()
error_df['Predicted_W']=final_pred
error_df['Absolute_Error']=(error_df.W-error_df.Predicted_W).abs()
error_df.sort_values('Absolute_Error',ascending=False).head(10)

## 9. Limitations, Ethics, and Reflection

The dataset is observational and aggregated at the team-season level, so the model cannot establish causation. The selected features omit defense, baserunning, injuries, roster construction, payroll, strength of schedule, park effects, bullpen usage, and other factors. The test period is only three seasons. Predictions are also uncertain and should not be treated as guarantees.

In this educational project, incorrect predictions have limited direct consequences. A real-world model used for betting, personnel, financial, or operational decisions could create larger consequences when predictions are wrong. Future work could add more features, use cross-validation within the training period, tune KNN's `k`, and compare additional algorithms.

## 10. Code and AI Transparency

Dataset: Lahman Baseball Database Teams table — https://github.com/corbtastik/lahman-baseball-db

Generative AI disclosure: OpenAI ChatGPT (GPT-5.6 Luna) assisted with project planning, code structure, model setup, interpretation, and written explanations. The student is responsible for reviewing and understanding the submitted code and results and for following course AI-use requirements.

## Conclusion

For the selected MLB team-season data, pitching/run prevention produced somewhat stronger standalone predictions of wins than the selected hitting variables, while the combined hitting-and-pitching models performed best. The final KNN model achieved the strongest held-out performance among the tested approaches. These results are predictive rather than causal and depend on the selected features and seasons.